# 02 – Exploratory Data Analysis (EDA)

> **Do AI Hype Cycles Create Temporary Stock Market Bubbles?**

This notebook explores the relationship between AI-related public interest (hype) and technology stock prices.
Hype is measured using a **combined index** of Google Trends search interest and Reddit mention counts.

### Structure
1. Setup and data loading
2. Descriptive statistics
3. Time series: stock prices + hype index with AI event overlays
4. Google Trends vs Reddit comparison
5. Hype vs returns — scatter and correlation matrix
6. Return distributions: hype vs normal periods
7. Lag analysis (does today's hype predict tomorrow's return?)
8. Volatility during hype vs normal periods

## CELL 1: Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in ["notebook", "notebooks"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES        = PROJECT_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
COLORS = {
    "NVDA":   "#76b900",
    "MSFT":   "#00a4ef",
    "AAPL":   "#555555",
    "hype":   "#e74c3c",
    "google": "#4285F4",
    "reddit": "#FF4500",
}
TICKERS = ["NVDA", "MSFT", "AAPL"]
print("Setup complete.")

## CELL 2: Load Data

In [ ]:
df     = pd.read_csv(DATA_PROCESSED / "merged_data.csv",  index_col="date", parse_dates=True)
events = pd.read_csv(DATA_PROCESSED / "ai_events.csv",    parse_dates=["date"])

print(f"Merged data shape : {df.shape}")
print(f"Date range        : {df.index.min().date()} → {df.index.max().date()}")
print(f"Hype days         : {df['is_hype_period'].sum()} ({df['is_hype_period'].mean()*100:.1f}%)")
print(f"Columns           : {df.columns.tolist()}")
df.head()

## CELL 3: Descriptive Statistics

In [ ]:
key_cols = [
    "NVDA_close", "MSFT_close", "AAPL_close",
    "NVDA_return", "MSFT_return", "AAPL_return",
    "google_hype_index", "reddit_hype_index", "hype_index"
]
print("── Descriptive Statistics ──────────────────────────────")
display(df[key_cols].describe().round(4))

print("\n── Mean Daily Returns: Hype vs Normal Periods ──────────")
for ticker in TICKERS:
    col = f"{ticker}_return"
    hype_mean   = df.loc[df["is_hype_period"] == 1, col].mean()
    normal_mean = df.loc[df["is_hype_period"] == 0, col].mean()
    print(f"  {ticker}: hype = {hype_mean*100:.3f}%/day  |  normal = {normal_mean*100:.3f}%/day")

## CELL 4: Time Series – Stock Prices and Combined Hype Index

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.suptitle("Stock Prices and AI Hype Index (2020–2024)", fontsize=14, fontweight="bold", y=1.01)

for ax, ticker in zip(axes[:3], TICKERS):
    ax.plot(df.index, df[f"{ticker}_close"], color=COLORS[ticker], linewidth=1.2, label=ticker)
    hype_mask = df["is_hype_period"] == 1
    ax.fill_between(df.index, 0, df[f"{ticker}_close"].max(),
                    where=hype_mask, alpha=0.10, color=COLORS["hype"], label="Hype period")
    for _, row in events.iterrows():
        if df.index.min() <= row["date"] <= df.index.max():
            ax.axvline(row["date"], color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.set_ylabel(f"{ticker} Price (USD)")
    ax.legend(loc="upper left", fontsize=9)

# Combined hype index panel
axes[3].fill_between(df.index, df["hype_index"], alpha=0.3, color=COLORS["hype"])
axes[3].plot(df.index, df["hype_index"],        color=COLORS["hype"],   linewidth=1.2, label="Combined")
axes[3].plot(df.index, df["google_hype_index"], color=COLORS["google"], linewidth=0.8, alpha=0.7, linestyle="--", label="Google")
axes[3].plot(df.index, df["reddit_hype_index"], color=COLORS["reddit"], linewidth=0.8, alpha=0.7, linestyle=":",  label="Reddit")

for _, row in events.iterrows():
    if df.index.min() <= row["date"] <= df.index.max():
        axes[3].axvline(row["date"], color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        axes[3].text(row["date"], 0.95, row["event"], rotation=90, fontsize=7, ha="right", color="gray",
                     transform=axes[3].get_xaxis_transform())

axes[3].set_ylabel("AI Hype Index")
axes[3].set_xlabel("Date")
axes[3].legend(fontsize=8, loc="upper left")
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(FIGURES / "01_timeseries_prices_hype.png", bbox_inches="tight")
plt.show()
print("✓ Figure saved.")

## CELL 5: Google Trends vs Reddit – Hype Signal Comparison

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle("AI Hype Signals: Google Trends vs Reddit Mentions", fontsize=13, fontweight="bold")

axes[0].fill_between(df.index, df["google_hype_index"], alpha=0.4, color=COLORS["google"])
axes[0].plot(df.index, df["google_hype_index"], color=COLORS["google"], linewidth=1)
axes[0].set_ylabel("Google Trends Hype Index")
axes[0].set_title("Google Trends (AI, ChatGPT, ML, Nvidia AI)")

axes[1].fill_between(df.index, df["reddit_hype_index"], alpha=0.4, color=COLORS["reddit"])
axes[1].plot(df.index, df["reddit_hype_index"], color=COLORS["reddit"], linewidth=1)
axes[1].set_ylabel("Reddit Hype Index")
axes[1].set_title("Reddit Mentions (r/MachineLearning, r/artificial, r/stocks, r/investing)")
axes[1].set_xlabel("Date")

for ax in axes:
    for _, row in events.iterrows():
        if df.index.min() <= row["date"] <= df.index.max():
            ax.axvline(row["date"], color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig(FIGURES / "02_google_vs_reddit_hype.png", bbox_inches="tight")
plt.show()

r = df[["google_hype_index", "reddit_hype_index"]].corr().iloc[0, 1]
print(f"Pearson correlation between Google and Reddit hype: r = {r:.4f}")

## CELL 6: Normalized Price Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for ticker in TICKERS:
    col = f"{ticker}_close"
    normalized = df[col] / df[col].iloc[0] * 100
    ax.plot(df.index, normalized, label=ticker, color=COLORS[ticker], linewidth=1.5)

ax.fill_between(df.index, 0, 2000,
                where=df["is_hype_period"]==1,
                alpha=0.08, color=COLORS["hype"], label="Hype period")

for _, row in events.iterrows():
    if df.index.min() <= row["date"] <= df.index.max():
        ax.axvline(row["date"], color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.text(row["date"], ax.get_ylim()[1]*0.92, row["event"],
                rotation=90, fontsize=7, ha="right", color="gray")

ax.set_title("Normalized Stock Prices (Base = 100 at 2020-01-01)", fontsize=13)
ax.set_ylabel("Indexed Price")
ax.set_xlabel("Date")
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "03_normalized_prices.png", bbox_inches="tight")
plt.show()

## CELL 7: Correlation Matrix

In [ ]:
corr_cols = [
    "hype_index", "google_hype_index", "reddit_hype_index",
    "NVDA_return", "MSFT_return", "AAPL_return",
    "NVDA_volatility", "MSFT_volatility", "AAPL_volatility"
]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax
)
ax.set_title("Correlation Matrix: Hype Indices vs Stock Returns & Volatility", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "04_correlation_matrix.png", bbox_inches="tight")
plt.show()

print("\nHype index correlations with returns:")
for hype_col in ["hype_index", "google_hype_index", "reddit_hype_index"]:
    print(f"\n  {hype_col}:")
    for ticker in TICKERS:
        r = corr_matrix.loc[hype_col, f"{ticker}_return"]
        print(f"    ↔ {ticker}_return: r = {r:.4f}")

## CELL 8: Hype vs Daily Returns – Scatter

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Combined AI Hype Index vs Daily Stock Returns", fontsize=13, fontweight="bold")

for ax, ticker in zip(axes, TICKERS):
    x = df["hype_index"]
    y = df[f"{ticker}_return"] * 100
    valid = pd.concat([x, y], axis=1).dropna()

    ax.scatter(valid.iloc[:, 0], valid.iloc[:, 1], alpha=0.25, s=8, color=COLORS[ticker])

    m, b = np.polyfit(valid.iloc[:, 0], valid.iloc[:, 1], 1)
    x_line = np.linspace(valid.iloc[:, 0].min(), valid.iloc[:, 0].max(), 100)
    ax.plot(x_line, m * x_line + b, color="black", linewidth=1.5, linestyle="--")

    r = valid.corr().iloc[0, 1]
    ax.set_title(f"{ticker}  (r = {r:.3f})", fontsize=11)
    ax.set_xlabel("AI Hype Index")
    ax.set_ylabel("Daily Return (%)")

plt.tight_layout()
plt.savefig(FIGURES / "05_hype_vs_returns_scatter.png", bbox_inches="tight")
plt.show()

## CELL 9: Return Distributions – Hype vs Normal

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Return Distributions: Hype Period vs Normal Period", fontsize=13, fontweight="bold")

for ax, ticker in zip(axes, TICKERS):
    col = f"{ticker}_return"
    hype_ret   = df.loc[df["is_hype_period"] == 1, col] * 100
    normal_ret = df.loc[df["is_hype_period"] == 0, col] * 100

    ax.hist(normal_ret.dropna(), bins=60, alpha=0.6, label="Normal", color="steelblue", density=True)
    ax.hist(hype_ret.dropna(),   bins=60, alpha=0.6, label="Hype",   color=COLORS["hype"],   density=True)
    ax.axvline(hype_ret.mean(),   color=COLORS["hype"],  linestyle="--", linewidth=1.5)
    ax.axvline(normal_ret.mean(), color="steelblue", linestyle="--", linewidth=1.5)

    ax.set_title(f"{ticker}", fontsize=11)
    ax.set_xlabel("Daily Return (%)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "06_return_distributions.png", bbox_inches="tight")
plt.show()

## CELL 10: Lag Analysis – Does Today's Hype Predict Tomorrow's Return?

In [ ]:
LAGS = range(0, 11)
lag_results = {ticker: [] for ticker in TICKERS}

for ticker in TICKERS:
    for lag in LAGS:
        r = df[f"{ticker}_return"].corr(df["hype_index"].shift(lag))
        lag_results[ticker].append(r)

fig, ax = plt.subplots(figsize=(10, 5))
for ticker in TICKERS:
    ax.plot(list(LAGS), lag_results[ticker], marker="o", label=ticker, color=COLORS[ticker])

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Lag (days): hype shifted N days into past")
ax.set_ylabel("Pearson Correlation with Daily Return")
ax.set_title("Cross-Correlation: AI Hype (lagged) vs Stock Returns", fontsize=12)
ax.set_xticks(list(LAGS))
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "07_lag_correlation.png", bbox_inches="tight")
plt.show()

lag_df = pd.DataFrame(lag_results, index=list(LAGS))
lag_df.index.name = "lag_days"
print("Lag correlation table:")
print(lag_df.round(4))

## CELL 11: Volatility – Hype vs Normal Periods

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(TICKERS))
width = 0.35

hype_vols   = [df.loc[df["is_hype_period"]==1, f"{t}_volatility"].mean() for t in TICKERS]
normal_vols = [df.loc[df["is_hype_period"]==0, f"{t}_volatility"].mean() for t in TICKERS]

bars1 = ax.bar(x - width/2, normal_vols, width, label="Normal period", color="steelblue", alpha=0.8)
bars2 = ax.bar(x + width/2, hype_vols,   width, label="Hype period",   color=COLORS["hype"], alpha=0.8)

ax.bar_label(bars1, fmt="%.3f", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.3f", padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(TICKERS)
ax.set_ylabel("Annualised Volatility (20-day rolling mean)")
ax.set_title("Stock Volatility: Hype Period vs Normal Period", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "08_volatility_comparison.png", bbox_inches="tight")
plt.show()